In [1]:
# Данный ноутбук использовал окружение google-colab
%pip install catboost fasttext -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.4/73.4 kB 3.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 MB 8.6 MB/s eta 0:00:00


# Домашнее задание "NLP. Часть 1"

In [2]:
import math
import re
import os
import random
import json
from collections import Counter, defaultdict
from typing import List, Dict, Tuple, Any

import torch
import numpy as np
import datasets
import fasttext
import fasttext.util
from transformers import BertTokenizer, BertModel

In [3]:
def seed_everything(seed: int):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True

seed_everything(42)

In [4]:
def normalize_pretokenize_text(text: str) -> List[str]:
    text = text.lower()
    words = re.findall(r'\b\w+\b', text)
    return words

In [5]:
# This block is for tests only
test_corpus = [
    "the quick brown fox jumps over the lazy dog",
    "never jump over the lazy dog quickly",
    "brown foxes are quick and dogs are lazy"
]

def build_vocab(texts: List[str]) -> Tuple[List[str], Dict[str, int]]:
    all_words = []
    for text in texts:
        words = normalize_pretokenize_text(text)
        all_words.extend(words)
    vocab = sorted(set(all_words))
    vocab_index = {word: idx for idx, word in enumerate(vocab)}
    return vocab, vocab_index

vocab, vocab_index = build_vocab(test_corpus)

## Задание 1 (0.5 балла)
Реализовать One-Hot векторизацию текстов

In [6]:
def one_hot_vectorization(
    text: str,
    vocab: List[str] = None,
    vocab_index: Dict[str, int] = None
) -> List[List[int]]:
    words = normalize_pretokenize_text(text)
    n_dim = len(vocab_index)
    result = []
    for word in words:
      vector = np.zeros(n_dim, dtype=np.int32)
      vector[vocab_index[word]] = 1
      result.append(vector.tolist())
    return result

def test_one_hot_vectorization(
    vocab: List[str],
    vocab_index: Dict[str, int]
) -> bool:
    try:
        text = "the quick brown fox"
        result = one_hot_vectorization(text, vocab, vocab_index)

        if not isinstance(result, list):
            return False

        expected_length = len(vocab)
        if len(result[0]) != expected_length:
            return False

        words_in_text = normalize_pretokenize_text(text)
        for i, word in enumerate(words_in_text):
            if word in vocab_index:
                idx = vocab_index[word]
                if result[i][idx] != 1:
                    return False

        print("One-Hot-Vectors test PASSED")

        return True
    except Exception as e:
        print(f"One-Hot-Vectors test FAILED: {e}")
        return False

In [7]:
assert test_one_hot_vectorization(vocab, vocab_index)

One-Hot-Vectors test PASSED


## Задание 2 (0.5 балла)
Реализовать Bag-of-Words

In [8]:
def bag_of_words_vectorization(text: str) -> Dict[str, int]:
    result = defaultdict(int)
    words = normalize_pretokenize_text(text)
    for word in words:
      result[word] += 1
    return result

def test_bag_of_words_vectorization() -> bool:
    try:
        text = "the the quick brown brown brown"
        result = bag_of_words_vectorization(text)

        if not isinstance(result, dict):
            return False

        if result.get('the', 0) != 2:
            return False
        if result.get('quick', 0) != 1:
            return False
        if result.get('brown', 0) != 3:
            return False
        if result.get('nonexistent', 0) != 0:
            return False

        print("Bad-of-Words test PASSED")
        return True
    except Exception as e:
        print(f"Bag-of-Words test FAILED: {e}")
        return False

In [9]:
assert test_bag_of_words_vectorization()

Bad-of-Words test PASSED


## Задание 3 (0.5 балла)
Реализовать TF-IDF

In [16]:
def tf_idf_vectorization(text: str, corpus: List[str] = None, vocab: List[str] = None, vocab_index: Dict[str, int] = None) -> List[float]:
    words = normalize_pretokenize_text(text)
    tf = defaultdict(float)
    for word in words:
      tf[word] += 1
    for k, v in tf.items():
      tf[k] = v / len(words)

    idf = defaultdict(float)
    for word in words:
      for doc in corpus:
        if word in doc:
          idf[word] += 1
    for k, v in idf.items():
      idf[k] = math.log(len(corpus) / v)

    result = np.zeros(len(vocab), dtype=np.float32)
    for word in words:
      result[vocab_index[word]] = tf[word] * idf[word]
    return result.tolist()

def test_tf_idf_vectorization(corpus, vocab, vocab_index) -> bool:
    try:
        text = "the quick brown"
        result = tf_idf_vectorization(text, corpus, vocab, vocab_index)

        if not isinstance(result, list):
            return False

        expected_length = len(vocab)
        if len(result) != expected_length:
            return False

        for val in result:
            if not isinstance(val, float):
                return False

        print("TF-IDF test PASSED")
        return True
    except Exception as e:
        print(f"TF-IDF test FAILED: {e}")
        return False

In [17]:
assert test_tf_idf_vectorization(test_corpus, vocab, vocab_index)

TF-IDF test PASSED


## Задание 4 (1 балл)
Реализовать Positive Pointwise Mutual Information (PPMI).  
https://en.wikipedia.org/wiki/Pointwise_mutual_information
$$PPMI(word, context) = max(0, PMI(word, context))$$
$$PMI(word, context) = log \frac{P(word, context)}{P(word) P(context)} = log \frac{N(word, context)|(word, context)|}{N(word) N(context)}$$
где $N(word, context)$ -- число вхождений слова $word$ в окно $context$ (размер окна -- гиперпараметр)

In [40]:
def ppmi_vectorization(
    text: str,
    corpus: List[str] = None,
    vocab: List[str] = None,
    vocab_index: Dict[str, int] = None,
    window_size: int = 2
) -> List[float]:
    cooccurrence_counts = defaultdict(int)
    for text in corpus:
      words = normalize_pretokenize_text(text)
      for i in range(len(words)):
        next_tokens = words[i+1 : i+1+window_size]
        for t in next_tokens:
          key = tuple(sorted([t, words[i]]))
          cooccurrence_counts[key] += 1
    cooccurrence = np.zeros((len(vocab), len(vocab)), dtype=np.float32)
    for k, v in cooccurrence_counts.items():
      cooccurrence[vocab_index[k[0]], vocab_index[k[1]]] = v
      cooccurrence[vocab_index[k[1]], vocab_index[k[0]]] = v

    col_totals = cooccurrence.sum(axis=0)
    total = col_totals.sum()
    row_totals = cooccurrence.sum(axis=1)
    denominator = np.outer(row_totals, col_totals)
    numerator = cooccurrence * total
    pmi = numerator / denominator
    with np.errstate(divide='ignore'):
      pmi = np.log(pmi)
    # log(0) = 0
    pmi[np.isinf(pmi)] = 0.0
    # making ppmi
    pmi[pmi < 0] = 0.0
    ppmi_vector = pmi.sum(axis=1)
    return ppmi_vector.tolist()


def test_ppmi_vectorization(corpus, vocab, vocab_index) -> bool:
    try:
        text = "quick brown fox"
        result = ppmi_vectorization(text, corpus, vocab, vocab_index)

        if not isinstance(result, list):
            return False

        expected_length = len(vocab)
        if len(result) != expected_length:
            return False

        for val in result:
            if not isinstance(val, float):
                return False

        print("PPMI test PASSED")
        return True
    except Exception as e:
        print(f"PPMI test FAILED: {e}")
        return False

In [41]:
assert test_ppmi_vectorization(test_corpus, vocab, vocab_index)

PPMI test PASSED


## Задание 5 (1 балл)
Реализовать получение эмбеддингов из fasttext и bert (для bert лучше использовать CLS токен)

In [45]:
def get_fasttext_embeddings(text: str, model_path: str = None, model: any = None) -> List[np.ndarray]:
    if model is None:
      if model_path is not None:
        model = fasttext.load_model(model_path)
      else:
        fasttext.util.download_model('en', if_exists='ignore')  # English
        model = fasttext.load_model('cc.en.300.bin')

    words = normalize_pretokenize_text(text)
    embeds = []
    for word in words:
      vector = model.get_word_vector(word)
      embeds.append(vector)
    return embeds


In [46]:
get_fasttext_embeddings("bro what the hell")

[array([ 0.24474624, -0.04772904, -0.03660253,  0.04744088, -0.17838906,
        -0.2126024 ,  0.20675084,  0.00653114,  0.06532399, -0.00255749,
        -0.17120343,  0.056776  , -0.20163731, -0.06428926,  0.03551618,
         0.10650854,  0.05354635,  0.03368896, -0.10032377,  0.07955564,
         0.0651658 ,  0.18002103,  0.00447479,  0.03832015, -0.00841149,
        -0.03063283,  0.03268043, -0.00825978, -0.01205749,  0.27220353,
        -0.16129808,  0.12735735, -0.13917278,  0.10030484,  0.11628117,
         0.04688274,  0.06425583, -0.0535143 , -0.10629866,  0.15196562,
         0.10432743,  0.03634842, -0.24783291, -0.05397034, -0.02789463,
         0.08641911, -0.02884251, -0.03742729, -0.03973718,  0.15511213,
         0.1476507 ,  0.13299339, -0.0794303 ,  0.09762006, -0.16443115,
        -0.034874  , -0.09879933,  0.0515123 , -0.06755264,  0.18643895,
         0.05654438,  0.03732143, -0.24363261,  0.16712084,  0.13293107,
        -0.03082484, -0.13970228,  0.36676526, -0.1

In [50]:
def get_bert_embeddings(
    text: str,
    model_name: str = 'bert-base-uncased',
    pool_method: str = 'cls'
) -> np.ndarray:
    tokenizer = BertTokenizer.from_pretrained(model_name)

    model = BertModel.from_pretrained(model_name)
    model.eval()

    encoding = tokenizer(
        text,
        padding=True,
        truncation=True,
        return_tensors="pt",
        add_special_tokens=True,
        max_length=512
    )

    input_ids = encoding["input_ids"]
    attention_mask = encoding["attention_mask"]
    with torch.no_grad():
      outputs = model(input_ids, attention_mask=attention_mask)
      word_embeds = outputs.last_hidden_state.mean(dim=1)
    return word_embeds


In [51]:
get_bert_embeddings("bro what the hell")

tensor([[ 1.9866e-01,  1.8902e-01, -4.0356e-02, -1.4189e-01, -7.0180e-02,
         -4.0922e-01, -1.0388e-01,  2.4918e-01, -3.0463e-02, -7.3954e-02,
         -6.2166e-02, -4.0749e-01, -2.3076e-01,  6.5374e-01, -3.0432e-01,
          3.1655e-01,  1.7456e-01, -2.2683e-01, -2.5997e-01,  3.2424e-01,
          1.7967e-01,  5.5076e-01, -6.3606e-01,  6.1540e-02,  3.0601e-01,
          3.4342e-01, -2.5208e-01, -2.0815e-01,  1.7832e-01,  3.7729e-02,
          1.0154e-01, -1.7269e-01, -1.3488e-01, -4.0342e-01, -1.5022e-01,
          8.9902e-02,  6.3707e-02, -1.9331e-01, -2.2304e-01, -7.9219e-02,
         -3.9746e-01, -1.0473e-01,  1.9418e-01,  1.5796e-01, -1.6189e-01,
         -3.5295e-01,  7.1819e-01, -1.7587e-01, -1.8880e-01, -2.5844e-01,
          1.2956e-01, -8.8385e-02, -7.2695e-01,  2.9850e-03, -1.5587e-01,
          1.8354e-01,  3.9615e-01, -4.5577e-01, -1.1899e-01,  3.2651e-01,
          4.0869e-01,  1.9826e-02,  6.7837e-02, -5.1617e-01,  6.6338e-01,
          3.7346e-01, -4.8756e-02,  4.

## Задание 6 (1.5 балла)
Реализовать обучение так, чтобы можно было поверх эмбеддингов, реализованных в предыдущих заданиях, обучить какую-то модель (вероятно неглубокую, например, CatBoost) на задаче классификации текстов ([IMDB](https://huggingface.co/datasets/stanfordnlp/imdb)).

In [74]:
def vectorize_dataset(
    dataset_name: str = "imdb",
    vectorizer_type: str = "bow",
    split: str = "train",
    sample_size: int = 2500
) -> Tuple[Any, List, List]:

    dataset = datasets.load_dataset(dataset_name, split=split)

    if sample_size:
        dataset.shuffle(seed=42)
        dataset = dataset.select(range(min(sample_size, len(dataset))))

    texts = [item['text'] for item in dataset if 'text' in item and item['text'].strip()]
    labels = [item['label'] for item in dataset if 'label' in item]

    def build_vocab(texts: List[str]) -> Tuple[List[str], Dict[str, int]]:
        all_words = []
        for text in texts:
            words = normalize_pretokenize_text(text)
            all_words.extend(words)
        vocab = sorted(set(all_words))
        vocab_index = {word: idx for idx, word in enumerate(vocab)}
        return vocab, vocab_index

    vocab, vocab_index = build_vocab(texts)

    vectorized_data = []
    for text in texts:
        if vectorizer_type == "one_hot":
            vectorized_data.append(one_hot_vectorization(text, vocab, vocab_index))
        elif vectorizer_type == "bow":
            bow_dict = bag_of_words_vectorization(text)
            vector = [bow_dict.get(word, 0) for word in vocab]
            vectorized_data.append(vector)
        elif vectorizer_type == "tfidf":
            vectorized_data.append(tf_idf_vectorization(text, texts, vocab, vocab_index))
        elif vectorizer_type == "ppmi":
            vectorized_data.append(ppmi_vectorization(text, texts, vocab, vocab_index))
        elif vectorizer_type == "fasttext":
            embeddings = get_fasttext_embeddings(text)
            if embeddings:
                avg_embedding = np.mean(embeddings, axis=0)
                vectorized_data.append(avg_embedding.tolist())
            else:
                vectorized_data.append([0] * 300)
        elif vectorizer_type == "bert":
            embedding = get_bert_embeddings(text)
            vectorized_data.append(embedding.tolist())
        else:
            raise ValueError(f"Unknown vectorizer type: {vectorizer_type}")
    return vocab, vectorized_data, labels

In [75]:
from catboost import CatBoostClassifier
from sklearn.metrics import classification_report, accuracy_score, f1_score
from sklearn.model_selection import train_test_split, cross_val_score, KFold

def train(
    embeddings_method="bow",
    test_size=0.2,
    val_size=0.2,
    cv_folds=5
):
    print(embeddings_method)
    vocab, X, y = vectorize_dataset("imdb", embeddings_method, "train")
    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=val_size, random_state=42)

    _, X_test, y_test = vectorize_dataset("imdb", embeddings_method, "test")

    model = CatBoostClassifier(verbose=False, random_state=42)

    model.fit(X_train, y_train, eval_set=(X_val, y_val), verbose=False)

    y_pred = model.predict(X_test)
    print(f"accuracy: {accuracy_score(y_pred, y_test)}")
    print(f"f1: {f1_score(y_pred, y_test)}")
    print(classification_report(y_pred, y_test))

In [76]:
for embeddings_method in ["bow", "one_hot", "tfidf", "ppmi", "fasttext", "bert"]:
    train(embeddings_method=embeddings_method)

bow


CatBoostError: catboost/private/libs/target/target_converter.cpp:404: Target contains only one unique value